# 07 Reaction Optimization: Buchwald-Hartwig Condition Search

This section uses real Buchwald-Hartwig data to show why reaction optimization is a sequential decision problem under limited budget.

Application scenario: when real experiments are expensive, chemists want to find high-yielding conditions within a limited number of experiments. The data comes from the Science 2018 work by Ahneman, Estrada, Lin, Dreher, and Doyle on predicting C-N cross-coupling reaction performance.

Reference:

- Ahneman, D. T.; Estrada, J. G.; Lin, S.; Dreher, S. D.; Doyle, A. G. Predicting reaction performance in C-N cross-coupling using machine learning. Science, 2018, 360, 186-190.

## Intuition

Reaction yield depends on substrate, ligand, base, additive, aryl halide, and related conditions. Because real experimental budgets are limited, we can train a surrogate model from observed experiments and ask it to suggest the next batch of conditions likely to give high yield.

Chemically, the ligand affects the metal center's electronic and steric environment; base and additive affect pathways and side reactions; aryl halide changes substrate reactivity. A model can learn associations from an existing table, but it cannot guarantee that a suggested condition is mechanistically reasonable.

## Mathematical and Chemical Definitions

A simple optimization loop:

```text
1. Observe a small set of experiments D = {(condition_i, yield_i)}
2. Train a surrogate: condition -> yield
3. Use an acquisition rule to select the next untested conditions
4. Read or run those experiments, update D
```

This section compares three strategy families:

1. random search: no learning; randomly sample conditions.
2. Random Forest greedy: train an RF surrogate on one-hot conditions and select the highest predicted mean.
3. Gaussian Process UCB: use a GP surrogate and select by `mean + beta * std`, balancing prediction and uncertainty.

To mimic high-throughput experimentation, this section uses batch optimization: each round selects 5 candidate conditions, waits until the whole batch is observed, then trains the next model.

Three initial designs are included:

- random initial design: randomly choose the first 5 conditions.
- Latin-like level coverage: cover different levels of each discrete factor as much as possible, similar to the LHC idea of covering dimensions.
- CVT-like/maximin: greedily choose points that are far apart in one-hot space, similar to space-filling or CVT-style coverage.

Strict LHC and CVT are mainly defined for continuous spaces. Reaction conditions here are discrete categorical variables, so these are categorical approximations.

The plot therefore contains 7 main curves: 1 random baseline plus `3 initial designs x 2 surrogate models`. Each curve uses 10 repeats and shows a mean with a 10-90% band, avoiding overinterpretation of one random seed.

## Read Buchwald-Hartwig Data

This cell reads the reaction-condition table and converts yield to numeric values. Each row is a measured condition combination. The simulation "reads" results from a fixed table; it does not run new wet-lab experiments.

In [ ]:
from pathlib import Path

START = Path.cwd().resolve()
for candidate in [START, *START.parents]:
    if (candidate / "data").exists() and (candidate / "notebooks").exists():
        ROOT = candidate
        break
else:
    raise FileNotFoundError(
        "Cannot find the materials root. Start Jupyter from the materials directory "
        "or from one of its subdirectories."
    )

DATA = ROOT / "data"
RAW = DATA / "raw"
EXAMPLES = DATA / "examples"
RANDOM_STATE = 42

print("materials root:", ROOT)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, DotProduct, WhiteKernel
from sklearn.preprocessing import OneHotEncoder

sns.set_theme(style="whitegrid", context="notebook")

# Each raw-table row is a reaction plus ligand/base/additive/aryl halide condition and the corresponding yield.
bh = pd.read_csv(RAW / "buchwald_hartwig.csv")
bh["yield"] = pd.to_numeric(bh["yield"], errors="coerce")
bh = bh.dropna(subset=["yield"]).reset_index(drop=True)
print(bh.shape)
bh.head()

## Define the Full Candidate Space

`reaction` represents substrate/reaction identity; ligand, additive, base, and aryl halide represent selectable reaction conditions. If a condition combination has repeated records, the mean yield is used so duplicates do not distort the optimization trace.

In [ ]:
condition_columns = ["reaction", "ligand", "additive", "base", "aryl halide"]

bh_conditions = bh[condition_columns + ["yield"]].dropna().copy()
for col in condition_columns:
    bh_conditions[col] = bh_conditions[col].astype(str)

# One candidate is a reaction plus condition combination; repeated measurements are represented by the mean yield.
candidate_df = (
    bh_conditions
    .groupby(condition_columns, as_index=False, sort=False)["yield"]
    .mean()
    .reset_index(drop=True)
)

# Count levels in each condition category to quantify why the combinatorial space grows quickly.
space_summary = pd.DataFrame(
    {
        "condition": condition_columns,
        "unique_choices": [candidate_df[col].nunique() for col in condition_columns],
    }
)
estimated_factorial = int(np.prod(space_summary["unique_choices"]))
print("realized candidate rows:", len(candidate_df))
print("full factorial combinations if fully crossed:", f"{estimated_factorial:,}")
space_summary

## Inspect Yield Distribution

If high-yielding conditions are rare, random search needs a larger budget to find them. If high-yielding conditions are common, random search can perform surprisingly well.

In [ ]:
plt.figure(figsize=(7, 4))
sns.histplot(candidate_df["yield"], bins=30)
plt.xlabel("yield")
plt.title("Buchwald-Hartwig yield distribution across realized candidates")
plt.tight_layout()

## How Rare Are High-Yielding Conditions?

If high-yielding conditions are not rare, random search is a strong baseline. This is why optimization studies must report random baselines carefully. The next cell checks the fraction of candidates above several yield thresholds.

In [ ]:
y_all = candidate_df["yield"].to_numpy(dtype=float)
thresholds = [40, 50, 60, 70, 80]

pd.DataFrame(
    {
        "yield_threshold": thresholds,
        "fraction_of_candidates_at_or_above": [(y_all >= t).mean() for t in thresholds],
        "count": [(y_all >= t).sum() for t in thresholds],
    }
)

## Encode Reaction Conditions

scikit-learn models cannot directly process string categories. One-hot encoding turns each categorical level into a 0/1 feature: whether a row uses a particular ligand, base, reaction id, and so on.

This is a simple representation. It can express whether a condition appears, but it does not explicitly encode coordination chemistry, electronic effects, or substrate similarity.

In [ ]:
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
X_all = encoder.fit_transform(candidate_df[condition_columns].astype(str))
y_all = candidate_df["yield"].to_numpy(dtype=float)

print("X shape:", X_all.shape)
print("yield range:", f"{y_all.min():.1f} to {y_all.max():.1f}")
candidate_df.nlargest(5, "yield")[condition_columns + ["yield"]]

## Define Random Search and Initial Designs

In reaction optimization, the first experiments are often designed to cover different ligands, bases, additives, or substrate types. The functions below define a batch random baseline and random/Latin-like/CVT-like initial designs.

In [ ]:
def batch_best_trace(yields, observed_order, batch_size=5):
    # After each batch, record the best yield observed so far.
    observed_order = list(map(int, observed_order))
    counts = []
    best_values = []
    for end in range(batch_size, len(observed_order) + 1, batch_size):
        counts.append(end)
        best_values.append(float(np.max(yields[observed_order[:end]])))

    # If the final batch is smaller than batch_size, still record the final result.
    if counts and counts[-1] != len(observed_order):
        counts.append(len(observed_order))
        best_values.append(float(np.max(yields[observed_order])))
    elif not counts and observed_order:
        counts.append(len(observed_order))
        best_values.append(float(np.max(yields[observed_order])))

    return np.asarray(counts), np.asarray(best_values)


def simulate_random_batches(yields, budget=40, batch_size=5, repeats=200, seed=RANDOM_STATE):
    # Baseline: each round randomly selects batch_size unobserved candidates; the next round is still random and no model is trained.
    # This process only uses the observed set to avoid repeats; it does not use observed yields to guide the next round.
    rng = np.random.default_rng(seed)
    traces = []
    n = len(yields)
    budget = min(budget, n)
    for _ in range(repeats):
        order = rng.choice(n, size=budget, replace=False)
        _, trace = batch_best_trace(yields, order, batch_size=batch_size)
        traces.append(trace)
    return np.asarray(traces)


def random_initial_indices(total_size, size=5, seed=RANDOM_STATE):
    # Random design for the initial batch.
    rng = np.random.default_rng(seed)
    return list(map(int, rng.choice(total_size, size=min(size, total_size), replace=False)))


def latin_like_initial_indices(df, columns, size=8, seed=RANDOM_STATE):
    # Discrete Latin-like initialization: each step prefers candidates that cover more unseen categorical levels.
    rng = np.random.default_rng(seed)
    selected = []
    used_values = {col: set() for col in columns}
    pool = list(rng.permutation(len(df)))

    while len(selected) < min(size, len(df)) and pool:
        gains = []
        for idx in pool:
            row = df.iloc[idx]
            gains.append(sum(row[col] not in used_values[col] for col in columns))

        # Break ties randomly when gain is equal, avoiding dependence on raw-table order.
        best_gain = max(gains)
        tied_positions = [pos for pos, gain in enumerate(gains) if gain == best_gain]
        chosen_position = int(rng.choice(tied_positions))
        idx = int(pool.pop(chosen_position))
        selected.append(idx)
        for col in columns:
            used_values[col].add(df.iloc[idx][col])

    return selected


def maximin_initial_indices(X, size=8, seed=RANDOM_STATE):
    # CVT-like / space-filling idea: greedily select candidates farthest from existing points in one-hot space.
    rng = np.random.default_rng(seed)
    n = X.shape[0]
    size = min(size, n)
    first = int(rng.integers(n))
    selected = [first]
    min_dist = np.full(n, np.inf)

    while len(selected) < size:
        last = selected[-1]
        diff = X - X[last]
        dist = np.sqrt(np.einsum("ij,ij->i", diff, diff))
        min_dist = np.minimum(min_dist, dist)
        min_dist[selected] = -np.inf
        selected.append(int(np.argmax(min_dist)))

    return selected

## Define Surrogate Optimization Loops

RF greedy only uses predicted mean and can exploit too early. GP-UCB uses both predicted mean and uncertainty:

```text
acquisition(x) = mean(x) + beta * std(x)
```

Larger `beta` encourages exploration of uncertain regions.

Each round selects a whole batch of candidates. The 5 points in one batch are selected simultaneously; the result of point 1 cannot update the choice of points 2-5 within the same batch.

In [ ]:
def _prepare_observed(init_indices, total_size, limit):
    # Deduplicate and ensure the initialization size does not exceed batch size, budget, or candidate count.
    observed = []
    for idx in init_indices:
        idx = int(idx)
        if idx not in observed:
            observed.append(idx)
        if len(observed) >= min(total_size, limit):
            break
    return observed


def _take_top_k(remaining, scores, k):
    # Select the k remaining candidates with the highest acquisition scores as the next experimental batch.
    order = np.argsort(scores)[::-1][:k]
    return [remaining[int(pos)] for pos in order]


def run_rf_batch_greedy(X, y, init_indices, budget=40, batch_size=5, seed=RANDOM_STATE):
    observed = _prepare_observed(init_indices, len(y), min(batch_size, budget))
    remaining = [idx for idx in range(len(y)) if idx not in observed]

    while len(observed) < min(budget, len(y)) and remaining:
        model = RandomForestRegressor(
            n_estimators=60,
            min_samples_leaf=2,
            random_state=seed,
            n_jobs=-1,
        )
        model.fit(X[observed], y[observed])
        predicted_mean = model.predict(X[remaining])

        # Greedy acquisition: each round selects a whole batch with the highest predicted means.
        k = min(batch_size, budget - len(observed), len(remaining))
        next_batch = _take_top_k(remaining, predicted_mean, k)
        observed.extend(next_batch)
        remaining = [idx for idx in remaining if idx not in set(next_batch)]
    counts, trace = batch_best_trace(y, observed, batch_size=batch_size)
    return counts, trace, observed


def run_gp_batch_ucb(X, y, init_indices, budget=40, batch_size=5, beta=1.5, seed=RANDOM_STATE):
    observed = _prepare_observed(init_indices, len(y), min(batch_size, budget))
    remaining = [idx for idx in range(len(y)) if idx not in observed]

    while len(observed) < min(budget, len(y)) and remaining:
        # DotProduct kernel on one-hot categorical features is similar to learning linear similarity;
        # WhiteKernel represents observation noise. Hyperparameter optimization is disabled for speed and stability.
        kernel = ConstantKernel(1.0) * DotProduct(sigma_0=1.0) + WhiteKernel(noise_level=1.0)
        model = GaussianProcessRegressor(
            kernel=kernel,
            normalize_y=True,
            alpha=1e-6,
            optimizer=None,
            random_state=seed,
        )
        model.fit(X[observed], y[observed])
        predicted_mean, predicted_std = model.predict(X[remaining], return_std=True)

        # UCB acquisition: exploitation + exploration。
        acquisition = predicted_mean + beta * predicted_std
        k = min(batch_size, budget - len(observed), len(remaining))
        next_batch = _take_top_k(remaining, acquisition, k)
        observed.extend(next_batch)
        remaining = [idx for idx in remaining if idx not in set(next_batch)]
    counts, trace = batch_best_trace(y, observed, batch_size=batch_size)
    return counts, trace, observed

## Compare Search Strategies

Each curve shows the best yield observed after each batch round. The black dashed line is the global best yield in this fixed table; in real experiments this line is usually unknown.

The random baseline means "select 5 unobserved candidates randomly each round, and remain random in the next round". It does not train a model, but it avoids repeated candidates. Surrogate methods use random/Latin-like/CVT-like design for the first 5 points, then use RF or GP-UCB to choose later batches.

To reduce randomness, each strategy runs 10 repeats. The x-axis is round: round 1 means the first batch of 5 experiments has completed, and round 20 corresponds to 100 total experiments.

If random search and surrogate search are close, possible reasons include:

1. The candidate space is not large relative to the budget.
2. High-yielding candidates are not rare.
3. One-hot categorical features are crude and do not encode reaction mechanism.
4. Early surrogate models are unstable with little observed data.

In [ ]:
batch_size = 5
n_rounds = 20
n_repeats = 10
budget = min(batch_size * n_rounds, len(candidate_df))
actual_rounds = int(np.ceil(budget / batch_size))
round_steps = np.arange(1, actual_rounds + 1)

def make_initial_design(design_name, seed):
    if design_name == "random init":
        return random_initial_indices(len(candidate_df), size=batch_size, seed=seed)
    if design_name == "Latin-like init":
        return latin_like_initial_indices(candidate_df, condition_columns, size=batch_size, seed=seed)
    if design_name == "CVT-like init":
        return maximin_initial_indices(X_all, size=batch_size, seed=seed)
    raise ValueError(f"unknown initial design: {design_name}")

initial_design_names = ["random init", "Latin-like init", "CVT-like init"]
random_traces = simulate_random_batches(
    y_all,
    budget=budget,
    batch_size=batch_size,
    repeats=n_repeats,
    seed=RANDOM_STATE,
)

method_results = {}
for design_name in initial_design_names:
    rf_traces = []
    rf_observed_runs = []
    gp_traces = []
    gp_observed_runs = []

    for repeat in range(n_repeats):
        seed = RANDOM_STATE + repeat
        init_indices = make_initial_design(design_name, seed)

        _, trace, observed = run_rf_batch_greedy(
            X_all, y_all, init_indices, budget=budget, batch_size=batch_size, seed=seed
        )
        rf_traces.append(trace)
        rf_observed_runs.append(observed)

        _, trace, observed = run_gp_batch_ucb(
            X_all, y_all, init_indices, budget=budget, batch_size=batch_size, beta=1.5, seed=seed
        )
        gp_traces.append(trace)
        gp_observed_runs.append(observed)

    rf_traces = np.asarray(rf_traces)
    gp_traces = np.asarray(gp_traces)
    rf_best_run = int(np.argmax(rf_traces[:, -1]))
    gp_best_run = int(np.argmax(gp_traces[:, -1]))

    method_results[f"RF + {design_name}"] = {
        "model": "RF greedy",
        "initial_design": design_name,
        "traces": rf_traces,
        "best_observed": rf_observed_runs[rf_best_run],
    }

    method_results[f"GP-UCB + {design_name}"] = {
        "model": "GP-UCB",
        "initial_design": design_name,
        "traces": gp_traces,
        "best_observed": gp_observed_runs[gp_best_run],
    }

plt.figure(figsize=(9, 5))
plt.plot(round_steps, random_traces.mean(axis=0), label="random baseline", linewidth=2.5, color="black")
plt.fill_between(
    round_steps,
    np.percentile(random_traces, 10, axis=0),
    np.percentile(random_traces, 90, axis=0),
    color="black",
    alpha=0.12,
    label="random 10-90%",
)

for label, result in method_results.items():
    linestyle = "--" if result["model"] == "RF greedy" else "-"
    traces = result["traces"]
    mean_trace = traces.mean(axis=0)
    p10 = np.percentile(traces, 10, axis=0)
    p90 = np.percentile(traces, 90, axis=0)
    line, = plt.plot(round_steps, mean_trace, label=label, linewidth=2, linestyle=linestyle)
    plt.fill_between(round_steps, p10, p90, color=line.get_color(), alpha=0.08)

plt.axhline(y_all.max(), linestyle="--", color="black", linewidth=1, label="best in dataset")
plt.xlabel("optimization round")
plt.ylabel("best yield observed so far")
plt.title(f"Batch reaction optimization: {batch_size} experiments/round, {n_repeats} repeats")
plt.xticks(round_steps)
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()

## Numerical Summary

The next table compares the final best observed values for random search and the 6 surrogate combinations across 10 repeats. The point is not to prove one method always wins, but to learn how to read mean performance and uncertainty bands.

If `RF + random init` is best in one run, that does not automatically mean it is stably better than GP or space-filling initialization. RF can be strong on one-hot categorical tables; GP-UCB uncertainty can also be poorly calibrated in sparse high-dimensional one-hot space. Use repeated means and 10-90% intervals before making a claim.

In [ ]:
random_final = random_traces[:, -1]

summary_rows = [
    {
        "method": "random baseline",
        "initial_design": "random every batch",
        "mean_best_yield_after_20_rounds": random_final.mean(),
        "p10": np.percentile(random_final, 10),
        "p90": np.percentile(random_final, 90),
        "best_repeat": random_final.max(),
    }
]

for label, result in method_results.items():
    final_values = result["traces"][:, -1]
    summary_rows.append(
        {
            "method": result["model"],
            "initial_design": result["initial_design"],
            "mean_best_yield_after_20_rounds": final_values.mean(),
            "p10": np.percentile(final_values, 10),
            "p90": np.percentile(final_values, 90),
            "best_repeat": final_values.max(),
        }
    )

method_summary = pd.DataFrame(summary_rows).sort_values("mean_best_yield_after_20_rounds", ascending=False)
method_summary

## Inspect Good Conditions Observed by Surrogates

Finally, list the top-yielding conditions observed in the best repeat of each surrogate combination. The analysis focus is not only whether a model won, but whether the suggested conditions are chemically plausible and whether further diversity should be explored when experiments are expensive.

In [ ]:
def best_observed_conditions(method_label, observed, top=3):
    out = candidate_df.iloc[observed].copy()
    out["order"] = np.arange(1, len(out) + 1)
    out["method"] = method_label
    return out.sort_values("yield", ascending=False).head(top)[["method", "order", *condition_columns, "yield"]]


pd.concat(
    [best_observed_conditions(label, result["best_observed"]) for label, result in method_results.items()],
    ignore_index=True,
)

## Observation Questions

1. Why can a surrogate loop look similar to random search when only one reaction id is fixed?
2. With batch size = 5, why can the first point in a batch not update the other four points?
3. What is the essential difference between RF greedy and GP-UCB acquisition?
4. Why are LHC/CVT-style space-filling designs more natural for continuous variables? How should they be interpreted for discrete reaction conditions?
5. If `RF + random init` is best in a single run, how can you decide whether that is chance or a stable advantage?
6. If a model-suggested condition conflicts with chemical intuition, should you trust the model or recheck data and assumptions?

### Hints

1. With a fixed reaction id, the space is small. If high-yielding conditions are not rare, random search can find good conditions quickly. Early RF models also have little data, and one-hot features are crude.
2. Batch experiments are submitted in parallel. The yields are known only after the whole batch finishes, so the model cannot update sequentially within the batch.
3. RF greedy uses only predicted mean and leans toward exploitation. GP-UCB adds predicted standard deviation, so uncertain regions can also be selected.
4. Continuous spaces have natural geometry for uniform coverage. Discrete categorical spaces lack a natural distance, so this tutorial approximates coverage by factor levels and one-hot distance.
5. Check the mean final result, percentile band, and curve shape across repeats. If intervals overlap heavily, do not overinterpret a single ranking.
6. First check data, encoding, and candidate feasibility. Treat the recommendation as a testable hypothesis; models do not replace chemical constraints or experimental safety.

## Summary

Buchwald-Hartwig data shows another common AI4Chem problem: not predicting one molecular property, but choosing next experiments under a limited budget. Random search is a strong baseline that must be taken seriously. GP, RF, and LHC/CVT-like initialization are tools for proposing experimental hypotheses more systematically. A surrogate model supports decisions; it does not replace chemical judgment.